In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if any(importlib.util.find_spec(name) is None
       for name in ("adh_ai_labs", "openai", "mcp")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "adh-ai-labs[llm] @ git+https://github.com/BradSegal/OxfordADH2026AILabs.git"], check=True)
    importlib.invalidate_caches()
print("Course helpers ready.")

# LLM integrations: a stroke-prevention audit with a language model

**MSc in Applied Digital Health.** About 50 mins.

Bradley Segal, Computational Health Informatics Lab, Institute of Biomedical Engineering, University of Oxford. Text released under CC BY 4.0, code under MIT licence.

In the LLM applications lab we ran a small language model on CPU. We saw how text is broken up into tokens, how the model provides a probability for every possible next token and that a conversation is simply a single long string of text that the model is completing. We also saw just how easy it was to mess this up as the 0.5B model gave a probability of 0.47 for ‘yes’ when given a note that stated ‘No evidence of atrial fibrillation’. In this lab we will move towards how we would use a bigger model in an application. We will use the model from code, solve a real-world task, and go through the end-to-end process.

A GP practice is performing an audit for stroke prevention in patients with atrial fibrillation (AF) according to NICE guidelines (NG196). They are interested in finding patients that may not have coded diagnoses of AF within their system. These patients have their diagnoses listed as free text within their clinic notes and as such are not available in a tabular format. The goal is to find these patients as part of the audit and determine their CHA2DS2-VASc score, a metric for determining stroke risk that is used to determine which patients should be offered anticoagulation. A complete audit would additionally review the treatment each patient is on, their risk of bleeding, as well as any decisions that have been made.

| Part | Time | What you do |
|---|---|---|
| Set-up | 6 min | Start a model on this Colab machine |
| 1. One call | 4 min | Use a programmatic interface to the LLM to send and receive messages, and test the model’s responses |
| 2. Notes to a table | 19 min | Tabulating notes, testing definitions on difficult notes, arbitrating disagreements, and refining definitions (TODOs 1 to 3) |
| 3. A calculator as a tool | 8 min | Outline the CHA2DS2-VASc calculator to ensure the model fills it in correctly (TODO 4) |
| 4. MCP | 8 min | Serve the calculator to any application then serve your own tool (TODO 5) |
| 5. Who should a clinician read? | 4 min | Count the patients each method finds and misses, and choose which notes a clinician reads (TODO 6) |
| Wrap-up | 1 min | Take-home points |

### How this notebook works

- Run each cell in order using **Shift + Enter**. If you have any issues, select *Runtime > Restart session and run all*.
- This notebook contains '**TODO**' cells which require you to complete between one and three lines of code. Some of the code has already been included to help you. Each 'TODO' cell has a corresponding check included which will print '✅' if correctly completed, '❌' with a hint if incorrectly completed, or '⏳' while a `...` is present or prior tasks are incomplete.
- **Try it** cells are experiments: feel free to change the text in quotes and re-run the cell as many times as you want.
- The model is not perfect and may get a case wrong even when your text is right. If the check still says so after a second attempt, move on.
- The **extensions** at the end are if you have any time left.

### Before you start

1. **GPU:** choose *Runtime > Change runtime type > T4 GPU*. If you don't have a GPU the notebook will still run but will use a smaller model on the CPU. Steps that need to read every note will read the first 16 live and then show a run completed on a GPU for the rest.
2. **Your copy:** *File > Save a copy in Drive*

The notes included in the course are synthetic and as such do not reflect a real patient. This means that you can freely send these notes to any model, unlike the ECG data from notebook 01.

## Set-up

The next cell will load the libraries, set up a model on this machine, and load the notes. This will take a minute or two to run the first time so run it now and read ahead while waiting for it to complete.

In [ ]:
import inspect
import json

import pandas as pd

from adh_ai_labs import extraction, llm, mcp_server, notes, tool_use
from adh_ai_labs import integration_figures as figures
from adh_ai_labs import integration_tables as tables
from adh_ai_labs.checks import llm_integrations as check

llm.start()
NOTES = notes.load()
print(f"{len(NOTES)} clinic notes loaded.")

`llm.start()` will install the open-source model host **Ollama** and download the model to the system. If you have a GPU this will be the Qwen3 4B Instruct model (Alibaba, Apache 2.0 License) which is a 4 billion parameter model that can respond in about a second. If you don’t have a GPU it will use the smaller Qwen2.5 1.5B Instruct model which is slower and tends to have more errors. The cell will show which one you have. The model will be running on this Colab machine and as such nothing you send to it will leave this system.

## Part 1. One call (4 min)

### The request every application sends

Within the LLM applications lab, you took a list of messages and using the chat template, converted it into a single string and then requested the model to predict the next token. This is not how applications typically perform this task. Instead, the list of messages is sent to a model **server**. This is a program which will apply the chat template and then run the model until the response is complete.

Your code interacts with the server via an application programming interface (**API**). This is a contract of how to ask for things and how they will be provided. The API format from OpenAI has become the standard. Ollama supports this and so do most commercial providers. As such, we can use the same code to interact with all services using the `openai` Python library.

The **client** is the object responsible for issuing your requests. You only need to specify where the server is.

In [ ]:
from openai import OpenAI

# A server on this machine checks no key, but the library needs one, so any text will do.
client = OpenAI(base_url=llm.URL, api_key="ollama")
print("The client sends requests to", llm.URL)

### One call

In this case, the messages are given **roles** as in the chat template: `system` for the application instructions and `user` for the query. The question asks the model if a particular patient within the audit has AF.

In [ ]:
note = NOTES.loc["N39", "note"]
messages = [{"role": "system", "content": "Answer yes or no only."},
            {"role": "user", "content": f"Does this patient have atrial fibrillation?\n\n{note}"}]
response = client.chat.completions.create(model=llm.MODEL, messages=messages, temperature=0)

tables.cards(NOTES, ["N39"])
print("The model's reply:", response.choices[0].message.content)
figures.request_response(messages, response)

The figure shows what happened to your request, filled in with your own call.

- `response.choices[0].message.content` is the response as text.
- `finish_reason` is `"stop"` if the model ended the turn and `"length"` if the model ran out of `max_tokens`.
- `usage` tracks the number of tokens read / written (the number a hosted provider will charge you for).
- `temperature=0` enforces that the model selects the most probable token at each step (greedy selection from the LLM applications lab). As such, the same query will almost always produce the same result.

The 4B model answers “No”. This seems to be answering about the patient’s current heart rhythm which is sinus rhythm, despite her AF diagnosis; for the audit the answer should be yes. The question is ambiguous: it does not say whether it means her rhythm today or her diagnosis.

### Try it: your own question

Change the question or the system message and run the cell again. The question below already asks about a diagnosis: does the answer change? To see how the model reads the question, set the system message to "Answer yes or no, then explain in one sentence." Can you find a question that gets the answer the audit needs?

In [ ]:
QUESTION = "Does this patient have a diagnosis of atrial fibrillation?"  # @param {type:"string"}
SYSTEM = "Answer yes or no only."  # @param {type:"string"}

reply = client.chat.completions.create(
    model=llm.MODEL, temperature=0,
    messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"{QUESTION}\n\n{note}"}])
print("Reply:", reply.choices[0].message.content)
print(f"Tokens read: {reply.usage.prompt_tokens}; written: {reply.usage.completion_tokens}")

## Part 2. Notes to a table (19 min)

### The notes and the audit's specification

The practice has extracted 41 notes from GP consultations, clinic letters, and discharge summaries. Here are three examples. The words relevant for AF have been highlighted.

In [ ]:
tables.cards(NOTES, ["N06", "N11", "N16"])

This was the specification the audit team wrote:

> **In the audit:** a patient with a diagnosis of atrial fibrillation of any type (paroxysmal, persistent or permanent), including a patient who is now in sinus rhythm.
>
> **Not in the audit:** suspected AF that has not been confirmed, a note that rules AF out, and atrial flutter.

By this specification, N06 is out: it’s her mother’s AF. N11 is in: she’s got paroxysmal AF but is in sinus rhythm today. N16 is out: atrial flutter is a different arrhythmia (NICE NG196 also suggests stroke risk assessment for atrial flutter; however, this audit is just for AF). These are the reasons why you need to be very specific about what is included in an audit.

### A keyword search first

Always try the simplest method before a model. It is the baseline the model has to beat. In the case of a practice, they would simply search for ‘AF’, ‘PAF’, or ‘atrial fibrillation’ within their notes.

- `str.contains` is the pandas method from the bootcamp.
- `|` means ‘or’.
- `\b` denotes a word boundary, hence `AF` does not match within “AFL”.

In [ ]:
keyword = NOTES["note"].str.contains(r"\bAF\b|\bPAF\b|atrial fibrillation", case=False)
print(f"The keyword search finds {keyword.sum()} of {len(NOTES)} notes.")

### From definitions to a table

The process involves writing the definitions for each of the columns (**fields**) in the table, in words. The figure shows what happens next.

In [ ]:
figures.extraction_step()

- Your definitions form the **system message**.
- They are then used to create a **JSON schema** which defines the various fields and their types (true or false, or text).
- The server constrains the LLM to only select tokens that will result in valid JSON. This is known as **constrained decoding** and leverages the next token probabilities from the LLM applications lab where all tokens that would violate the schema are zeroed out.
- Each note is sent as per Part 1 and the response is added as a row.

Here are some first draft definitions, the kind anyone would write. The cell reads all the notes with these definitions.

In [ ]:
FIELDS = {
    "atrial_fibrillation": "true if the patient has atrial fibrillation",
    "anticoagulant": "the anticoagulant the patient takes, or 'none', or 'not stated' if the note does not say",
}
table = extraction.extract_all(client, NOTES, FIELDS)
extraction.show(table, NOTES, keyword)

Scroll through the table and look for notes where the model’s yes or no does not match the specification, such as a mother’s AF, “?AF”, or a flutter.

### Try it: a definition on six hard notes

`extraction.try_definition` sends a single definition of `atrial_fibrillation` to six notes that are difficult for a variety of reasons. It visualises the model’s prediction against the label with a ✅ if they match. It only reads six notes so you can experiment with as many definitions as you desire.

In [ ]:
DEFINITION = "true if the patient has atrial fibrillation"  # @param {type:"string"}
extraction.try_definition(client, NOTES, DEFINITION)

### TODO 1: fields of your own

Try and add one or two fields of your own and see what the model can come up with for 8 notes. Some ideas could be:

- `date_of_diagnosis`: when AF was diagnosed;
- `rate_or_rhythm_drug`: a drug for rate or rhythm control, such as bisoprolol or flecainide;
- `bleeding_history`: any bleeding problem the note mentions.

Specify what the model should produce if the note does not contain such information. Then test what it does in the absence of such details. Does it follow your guidance or does it make something up (hallucinate)?

In [ ]:
MY_FIELDS = {
    "...": "...",  # a field name in lower case, then what the model should write
}
if check.own_fields(MY_FIELDS):
    own = extraction.try_fields(client, NOTES, MY_FIELDS)

### Where the model and the keyword search disagree

The model and keyword search are not the ‘truth’. Where they disagree, one of them must be wrong, so these notes are the first to review.

In [ ]:
chosen = extraction.disagreements(table, NOTES, keyword)

### TODO 2: who is right?

In the cell below, copy the `VERDICTS` line printed above. For each note, replace the `...` with:

- `"model"` if the model is correct by specification,
- `"keyword"` if the keyword search is correct,
- `"unclear"` if the specification does not settle it.

Then run the cell to see the labels.

In [ ]:
VERDICTS = ...  # paste the line printed above, then fill in each ...
if check.verdicts(VERDICTS, chosen):
    extraction.reveal(chosen, VERDICTS, table, NOTES, keyword)

The keyword search cannot read ‘excluded’, ‘mother had’, or ‘?AF’. The model can, but only as well as your definition tells it what to do with them.

### TODO 3: revise the definition

Refine the definition for `atrial_fibrillation` to ensure it accounts for the various types of notes it got incorrect. Specify what is true or false: a diagnosis now in sinus rhythm, suspected AF, ruled out AF, family member’s AF, flutter, etc. The cell runs this on the six challenging notes. Modify and rerun until satisfied with the result.

In [ ]:
DEFINITION_V2 = "true if the patient has atrial fibrillation ..."  # replace ... with what is true and what is false
FIELDS_V2 = None
if check.revision(DEFINITION_V2, FIELDS):
    FIELDS_V2 = {**FIELDS, "atrial_fibrillation": DEFINITION_V2}
    extraction.try_definition(client, NOTES, DEFINITION_V2)

Re-read all notes now with your refined definition and compare to the first draft.

In [ ]:
table_v2 = None
if check.ready(FIELDS_V2):
    table_v2 = extraction.extract_all(client, NOTES, FIELDS_V2, step="revised")
    extraction.before_after(table, table_v2, NOTES)

Did the answers for the notes you just reviewed change the way you meant? Did any other notes change? This is the process of prompt engineering for extraction. Write, review, refine, and measure the impact of each change. The smaller model on CPU is worse at following longer definitions and so refining the definition can make things worse.

## Part 3. A calculator as a tool (8 min)

### Why a tool

For every patient in the audit, NICE NG196 recommends that the CHA2DS2-VASc (Lip et al., *Chest*, 2010) score be calculated. The score adds points for each criterion below that the patient meets.

| Criterion | Points | What counts |
|---|---|---|
| **C**ongestive heart failure | 1 | Heart failure / weak left ventricle (reduced ejection fraction, e.g. “LVEF 35%”) |
| **H**ypertension | 1 | High blood pressure, even if controlled with treatment |
| **A**ge 75 or over | 2 | |
| **D**iabetes | 1 | Type 1 or 2, however it is treated |
| **S**troke | 2 | Prior stroke, transient ischaemic attack (TIA), or blood clot elsewhere (thromboembolism) |
| **V**ascular disease | 1 | Prior history of heart attack (myocardial infarction, MI), narrowed leg arteries (peripheral arterial disease, PAD), or the presence of aortic plaque |
| **A**ge 65 to 74 | 1 | |
| **S**ex **c**ategory | 1 | Female |

A language model is a token prediction machine. It can write a number that looks like a score; however, it is not reliable at arithmetic (see Extension C). Arithmetic is a job for ordinary code. Reading ‘LVEF 35%’ or ‘on ramipril for blood pressure’ and ticking the correct boxes is a task that a language model excels at.

A **tool** is simply a code function that the model is told about and can ask your code to run. The model does not get to see the code, simply a description of it. This is a combination of the tool’s name, description, as well as a listing of arguments that the tool requires, their format, and meaning. The model is then able to respond with a **tool call** which contains the tool name and the arguments it would like to run it with; your code is then responsible for running the actual function.

In [ ]:
figures.tool_loop()

This is the calculator, just a regular Python function. `True` is treated as 1 if you add it to a number.

In [ ]:
def cha2ds2_vasc(age: int, female: bool, heart_failure: bool, hypertension: bool, diabetes: bool,
                 stroke_tia: bool, vascular: bool) -> int:
    score = 2 if age >= 75 else 1 if age >= 65 else 0  # 75 or over: 2 points; 65 to 74: 1
    score += female + heart_failure + hypertension + diabetes + vascular  # 1 point each
    score += 2 * stroke_tia  # previous stroke, TIA or thromboembolism: 2 points
    return int(score)

The model will be used to score these three patients. These notes are synthetic and have their true scores provided. In reality, a risk factor not described in the note would necessitate checking the patient’s record and not assuming it to be absent.

In [ ]:
tables.cards(NOTES, ["N05", "N17", "N26"])

### What a vague description does

The `tool_use.describe` function constructs the description that the model sees. It takes the name and arguments from the function, but you write the description. In this case this is simply ‘Stroke risk score.’, with no description of each argument. The table shows the arguments the model chose beside the correct ones, and the score each gives.

In [ ]:
vague = tool_use.describe(cha2ds2_vasc, "Stroke risk score.")
results = tool_use.try_patients(client, vague, cha2ds2_vasc, NOTES, step="vague")

No errors were raised: every patient got a score. But in our GPU run the model gave Mr Bennett (N05) `heart_failure=False`, despite his ejection fraction of 35%, so his score was one point too low. A wrong score looks exactly like a right one.

### TODO 4: describe the tool

Most of the descriptions are written for you. Please write the missing two: `heart_failure` and `vascular`. Describe what counts in lay terms using the above table as reference.

In [ ]:
DESCRIPTION = "CHA2DS2-VASc stroke risk score (0 to 9) for a patient with atrial fibrillation."
PARAMETERS = {
    "age": "the patient's age in years",
    "female": "True if the patient is a woman (Mrs, Ms, a woman, F after the age), False for a man",
    "heart_failure": "...",  # what counts as heart failure?
    "hypertension": "high blood pressure, including when it is controlled on treatment",
    "diabetes": "diabetes mellitus, type 1 or type 2, however it is treated",
    "stroke_tia": "a previous stroke (including CVA or cerebral infarction), transient ischaemic attack (TIA) or "
                  "thromboembolism",
    "vascular": "...",  # what counts as vascular disease?
}

if check.tool_form(DESCRIPTION, PARAMETERS):
    tool = tool_use.describe(cha2ds2_vasc, DESCRIPTION, PARAMETERS)
    results = tool_use.try_patients(client, tool, cha2ds2_vasc, NOTES)
    check.tool_description(DESCRIPTION, PARAMETERS, results)

On the CPU, the smaller model can misread arguments even with good descriptions; the check then says so and shows the GPU run for comparison. Review the ❌ arguments and move on.

An **agent** is a program that lets a model use tools in a loop: it reads the result of each call and decides on the next action until the task is complete.

## Part 4. MCP (8 min)

### One plug for every application

In Part 3, the tool’s code and description were held within this notebook. The **Model Context Protocol (MCP)**, an open standard released at the tail-end of 2024, separates out these two components.

- The **tool server** holds the tools: it gives their descriptions and runs them.
- A **client** is any application that can connect to it. This can be a chat app, coding assistant, clinical system plug-in, or your own code.

The Python MCP package enables you to define a tool on a server by adding type hints (such as `age: int`) to a function and providing a **docstring** (the text in the triple quotes below the initial definition line). The server converts these to the description text read by the model. The diagram below shows the same loop as in Part 3 with the addition of the server.

In [ ]:
figures.mcp_loop()
print(inspect.getsource(mcp_server.cha2ds2_vasc))

The next cell initialises the server as a new process and requests the tools from the server. We then pass the model the note for Mrs Kowalski (N26) alongside the server’s tools. If the model calls a tool, we request the server to execute that tool and then return the results to the model. `await` indicates ‘wait for the server to respond’.

In [ ]:
async with mcp_server.connect() as server:
    listed = await server.list_tools()
for tool in listed.tools:
    print(f"The server offers {tool.name}: {tool.description.splitlines()[0]}")
print()

question = NOTES.loc["N26", "note"] + tool_use.QUESTION
system = tool_use.SYSTEM + " Answer in one sentence."
answer, calls = await mcp_server.ask(client, llm.MODEL, question, system=system, show_tools=False)
print("The model's answer:", answer)

You did not write a schema here! The server auto-generated it from the docstring. By connecting to a server, you agree to run its code and let it inject descriptions into your model's instructions. Only connect to servers you trust, just like any other software you install.

### TODO 5: a tool of your own

Apixaban, an anticoagulant medication, has a dose reduction in some patients. This rule is derived from the Apixaban (Eliquis) Summary of Product Characteristics, Section 4.2:

> The recommended dose of apixaban is 2.5 mg taken orally twice daily in patients with NVAF and at least two of the following characteristics: age ≥ 80 years, body weight ≤ 60 kg, or serum creatinine ≥ 1.5 mg/dL (133 micromole/L).

NVAF is non-valvular atrial fibrillation. Write the rule in a single line of Python, with a docstring that will be read by the model.

- A comparison such as `age >= 80` gives `True` or `False`.
- `True` counts as 1 when added, so adding the three comparisons counts the criteria met.

This cell tests your function against stored cases, deploys it to the MCP server, and provides the model with 2 short cases.

In [ ]:
def apixaban_reduced_dose(age: int, weight_kg: float, creatinine_umol_l: float) -> bool:
    """Whether a patient with atrial fibrillation should take the reduced apixaban dose (2.5 mg twice daily).

    age: ...
    weight_kg: ...
    creatinine_umol_l: ...
    """
    return ...  # True if at least two of: age 80 or over, weight 60 kg or less, creatinine 133 or more


if check.apixaban(apixaban_reduced_dose):
    own_results = await mcp_server.try_vignettes(client, llm.MODEL, [apixaban_reduced_dose])
    check.apixaban_tool(own_results)

Your function, type hints, and docstring are all the server needs. Any application that connects to this server can now use your rule.

## Part 5. Who should a clinician read? (4 min)

You can’t miss patients in the audit, because if you miss a patient, you don’t review their stroke risk. But what you can’t do is spend a clinician’s time reading all of the notes because that’s what they’re trying to avoid.

Here is the number of notes each method got correct and incorrect against the labels. The model is using your updated definition (on CPU, the first 16 notes; the rest are the run recorded on GPU).

In [ ]:
model_says_af = keyword_says_af = None
if check.ready(table_v2, task="TODO 3 and the cell that reads all the notes again"):
    model_says_af = extraction.says_af(table_v2)  # True or False for each note
    keyword_says_af = keyword
    figures.two_by_two({"Model, your revised definition": extraction.counts(model_says_af, NOTES),
                        "Keyword search": extraction.counts(keyword_says_af, NOTES)},
                       note=extraction.sources(table_v2))

### TODO 6: which notes go to a clinician?

Apply a decision rule to determine which notes should be reviewed by a clinician. Notes flagged by this rule are then read and correctly classified by a clinician; for the rest, the model’s answer stands. Combine the two methods' answers:

- `model_says_af | keyword_says_af`: either method says yes (`|` means or);
- `model_says_af != keyword_says_af`: they disagree;
- `model_says_af`: only the model's yeses.

The cell shows the outcome: how many patients are missed and how many notes the clinician has to review compared to all 41.

In [ ]:
REVIEW = ...  # choose one of the rules above
if check.ready(table_v2, task="TODO 3 and the cell that reads all the notes again") and check.review(REVIEW, NOTES):
    extraction.review(REVIEW, model_says_af, NOTES)

Try setting a different rule and rerunning the cell. Which would you advocate to the practice? The line under the table gives the share of the patients in the audit that your process finds (its **sensitivity**), with a 95% interval. The interval is wide because 41 notes is a small set. The estimate is also optimistic, because the definition was tuned on these notes. In practice this would be tested on a new set of notes that no one has seen prior to usage.

## Take-home points

- Communication to the model is via an API that allows for a list of messages to be submitted and returns a response with the model’s reply, the reason for the stop, and the number of tokens consumed. The same code can be used for both local and hosted models.
- Unstructured text extraction is only as good as the definitions you give it. Benchmark against a simple baseline, read the disagreements, and measure every revision.
- With a tool, the model reads the note and code does the arithmetic. The model’s only knowledge of the tool is the description and so vague descriptions result in plausible but incorrect answers.
- MCP enables a single server to define and execute tools for any application. Only connect to servers that you trust.
- Whether model performance is good enough should be gauged in relation to the cost of errors. In this scenario, a decision rule governing which notes to review requires consideration of the number of patients missed versus clinician time.

## Extensions (optional)

The first three ask what the model is worth; the rest are more technical, and the last prices a call.

### A. A judge for free text

Exact matching is effective when a question has a single correct answer, such as if a patient should be included in an audit. There is no single correct answer to what would be a patient-friendly explanation of a note. A common solution is to leverage a second model to grade model outputs based on a ‘**rubric**’, a series of written rules to grade by. This is known as an **LLM judge**. It is important to understand how well the judge follows the rules set out in the rubric. To do this you must first produce a rubric, then personally grade four explanations of a note and determine where the judge deviates. The four explanations are a good explanation, one with jargon, one that hallucinates a stroke, and one that suggests stopping an anticoagulant.

In [ ]:
from adh_ai_labs import bench

note = NOTES.loc[bench.EXPLAIN_NOTE, "note"]
tables.cards(NOTES, [bench.EXPLAIN_NOTE])
for number, text in enumerate(bench.EXPLANATIONS, start=1):
    print(f"{number}. {text}\n")

RUBRIC = ("You mark explanations of a clinic note written for the patient. An explanation is acceptable only if it "
          "says nothing the note does not say, uses no medical terms or abbreviations without explaining them, "
          "and gives no advice about treatment. Give a one-sentence reason, then your mark.")
MY_MARKS = [True, False, False, False]  # what your rubric says for explanations 1 to 4
marks = bench.judge_explanations(client, llm.MODEL, RUBRIC, note, bench.EXPLANATIONS)
print("\nDisagreements:", [n for n, (m, j) in enumerate(zip(MY_MARKS, marks), 1) if j is not None and m != j] or "none")
print("No usable judgement:", [n for n, j in enumerate(marks, 1) if j is None] or "none")

Test with a weak rubric (“Is this a good explanation?”) and determine what errors leak through.

### B. A hosted model

Given that the notes are synthetic, you are able to send these to a hosted model (never send ECG data from notebook 01). If you have an API key from an OpenAI compatible provider you can save this to your Colab secrets (key icon on left bar) as `API_KEY`. You can then set the address of the provider and the model you want to use. You can then run your updated extraction on a single note. You'll notice that the only thing that changes is the client. The schema remains the same as in Part 2 but is specified in the strict format that OpenAI's API requires to enforce it.

In [ ]:
BASE_URL = "https://api.openai.com/v1"  # or your provider's OpenAI-compatible address
HOSTED_MODEL = "gpt-6-luna"  # a model your key can use
try:
    from google.colab import userdata
    fields = FIELDS_V2 or FIELDS  # your revised definitions, if you have them
    hosted = OpenAI(base_url=BASE_URL, api_key=userdata.get("API_KEY"))
    reply = hosted.chat.completions.create(model=HOSTED_MODEL, temperature=0,
                                           messages=extraction.messages(NOTES.loc["N34", "note"], fields),
                                           response_format=extraction.response_format(fields, strict=True))
    print(reply.choices[0].message.content, reply.usage)
except Exception as error:  # no key, no Colab, or the provider refused
    print("No hosted call:", type(error).__name__, str(error)[:120])

### C. Without a tool

Prompt the model for the three patients’ scores without using a calculator and review the model’s reasoning.

In [ ]:
replies = tool_use.without_tool(client, NOTES)

### D. Tests for code a model writes

This task has the model writing the CHA2DS2-VASc calculator from scratch. The evaluation checks the code against various cases of patients with known scores. These cases are selected to detect common errors such as missing the vascular disease criterion. The evaluation then runs the cases against different implementations of the function each with a specific error introduced. The generated code is then evaluated in a subprocess with memory and time constraints, and with Python's network access blocked. This is a containment strategy and not a sandbox; the code still runs on the host machine.

In [ ]:
MY_TESTS = [
    bench.case("a man of 50 with no risk factors", 0, age=50),
    bench.case("a woman of 50 with no other risk factors", 1, age=50, female=True),
    bench.case("vascular disease alone", 1, age=50, vascular=True),
    bench.case("stroke or TIA alone", 2, age=50, stroke_tia=True),
    bench.case("aged 75", 2, age=75),
]
check.tests(MY_TESTS)
codes = {"from memory": bench.ask_for_code(client, llm.MODEL, bench.REQUEST),
         "with the rules": bench.ask_for_code(client, llm.MODEL, bench.REQUEST + bench.RULES)}
attempts = bench.test_code(codes, MY_TESTS)

Just because the code passes tests does not mean it is correct; it means that it matches your test cases.

### E. A multiple-choice benchmark

Most released medical benchmarks use multiple choice questions, such as MedMCQA: 194k questions from Indian medical entrance exams (Pal et al., 2022). This cell has both models attempt to answer 20 of these (Apache 2.0 licence; the licence file is included in the course package). It then evaluates the **exact match** for the final letter and produces a 95% interval as well as a per question analysis of the performance (McNemar’s exact test which only considers questions that one model got right and the other did not). If using a GPU it downloads the 1.5B model for comparison. If no model server is provided it will simply demonstrate one of the recorded models without comparisons.

In [ ]:
from scipy.stats import binomtest

models = bench.models_to_compare()
replies = bench.answer_questions(client, models)
if len(models) == 2:
    wide = replies.pivot(index="id", columns="model", values="exact").astype(bool)
    only_first = int((wide[models[0]] & ~wide[models[1]]).sum())
    only_second = int((~wide[models[0]] & wide[models[1]]).sum())
    if only_first + only_second:
        p = binomtest(only_first, only_first + only_second).pvalue
        print(f"Only {models[0]} right: {only_first}; only {models[1]} right: {only_second}; p = {p:.2g}")

20 questions cannot separate two models that are separated by a handful of points. Can you find a published medical leaderboard? Is it possible the models have seen the questions during training? How many questions are there? Are there intervals?

### F. A CURB-65 tool on the server

CURB-65 is a scoring tool for determining the severity of community-acquired pneumonia (NICE NG250). The tool provides a point for each of the following: new onset confusion, urea > 7mmol/L, respiratory rate ≥ 30, low blood pressure (systolic < 90 or diastolic ≤ 60mmHg), and age ≥ 65. Please read the docstring and function for the tool. The server constructs the tool description using the docstring. The model is then shown using the tool on two text-based cases.

In [ ]:
def curb_65(confusion: bool, urea: float, respiratory_rate: int, systolic_bp: int, diastolic_bp: int,
            age: int) -> int:
    """CURB-65 severity score (0 to 5) for an adult with community-acquired pneumonia.

    confusion: True if the patient is newly confused or disorientated.
    urea: blood urea in mmol/L.
    respiratory_rate: breaths per minute.
    systolic_bp: systolic blood pressure in mmHg, the first number of a reading such as 126/74.
    diastolic_bp: diastolic blood pressure in mmHg, the second number of a reading such as 126/74.
    age: the patient's age in years.
    """
    score = int(confusion)
    score += urea > 7
    score += respiratory_rate >= 30
    score += systolic_bp < 90 or diastolic_bp <= 60
    score += age >= 65
    return score


if check.curb_65_function(curb_65):
    curb_results = await mcp_server.try_vignettes(client, llm.MODEL, [curb_65])
    check.curb_65_tool(curb_results)

Remove the docstring explanation as to which number in a reading is systolic and which diastolic and re-run. Does the model still correctly split 104/58?

### G. Choosing a tool, and forcing one

The model can be configured to choose which tool to use if multiple are provided. `tool_choice` can be set to `"auto"` which is the default and allows the model to choose, `"none"` which prevents the model from using any tools, or the name of the function to force it to use a specific tool.

In [ ]:
async with mcp_server.connect() as server:
    server_tools = mcp_server.openai_tools((await server.list_tools()).tools)

for question in ["QT 400 ms at a heart rate of 75: what is the QTc?",
                 NOTES.loc["N26", "note"] + tool_use.QUESTION,
                 "What does the CHA2DS2-VASc score measure?"]:
    response = client.chat.completions.create(
        model=llm.MODEL, temperature=0, max_tokens=300, tools=server_tools,
        messages=[{"role": "user", "content": question}])
    calls = response.choices[0].message.tool_calls or []
    print(question.splitlines()[0][:60], "->", [call.function.name for call in calls] or "no tool")

Try forcing a tool by adding `tool_choice={"type": "function", "function": {"name": "qtc_bazett"}}` to the call and then inspect `tool_calls` for the 3rd question for which there is no tool available. Note that this version of Ollama may not respect this parameter as a common format for API requests doesn’t necessarily mean the same features are available.

### H. Streaming

Chat websites have the ability to visualise text as it is being generated. `stream=True` enables this by streaming responses in real-time from the server as they are generated.

In [ ]:
stream = client.chat.completions.create(
    model=llm.MODEL, temperature=0, max_tokens=120, stream=True,
    messages=[{"role": "user", "content": "In three sentences, why does atrial fibrillation raise stroke risk?"}])
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

### I. The context window

A model can only read so many tokens at a time; this is its **context window**. The server sets a limit on this window (this extension needs a GPU). The cell here submits a codeword, then submits all notes 6 times, and then requests the codeword. You can see `usage.prompt_tokens` compared to the length of text sent. If a prompt is too long, Ollama will simply truncate the beginning of the message without any indication. `usage` cannot inform you what was cut.

In [ ]:
everything = "\n\n".join(f"{i}: {text}" for i, text in NOTES["note"].items())
prompt = ("The audit's code word is HERON.\n\n" + (everything + "\n\n") * 6
          + "What is the audit's code word given at the very start? Answer in one word.")
if llm.ON_GPU:  # on the CPU, reading this many tokens takes several minutes
    response = client.chat.completions.create(model=llm.MODEL, temperature=0, max_tokens=30,
                                              messages=[{"role": "user", "content": prompt}])
    print(f"Sent about {len(prompt) // 4:,} tokens' worth of text; usage says {response.usage.prompt_tokens:,} "
          "were read.")
    print("Reply:", response.choices[0].message.content)

### J. What would a call like this cost?

The local model is free per call but someone is paying for the machine. The hosted providers charge per million tokens and the cost of writing a token is more than reading one. The cell below sends Part 1's call again and shows the cost of 1000 calls like it at the published rates (15th November 2026).

In [ ]:
first_call = client.chat.completions.create(model=llm.MODEL, messages=messages, temperature=0)  # Part 1's call again
prices = pd.DataFrame(llm.PRICES)
tokens_in, tokens_out = first_call.usage.prompt_tokens, first_call.usage.completion_tokens
prices["dollars_per_1000_calls"] = 1000 * (tokens_in * prices["input_per_million"]
                                           + tokens_out * prices["output_per_million"]) / 1_000_000
prices[["model", "provider", "dollars_per_1000_calls"]].round(3)

For a single practice’s notes these are all quite cheap; however, there is a 100x difference between the cheapest and most expensive model which is relevant when scaled to a health system.